In [ ]:
# ============================================================
# BLOCK 1: Import required libraries
# ============================================================
# ✅ Imports all needed packages
# Ensure you’ve already run:  pip install pandas numpy

import pandas as pd
import numpy as np
import google.generativeai as genai

from math import log1p, exp

print("✅ Libraries imported successfully.")


✅ Libraries imported successfully.


In [12]:
# ============================================================
# BLOCK 2: Load the mempool dataset safely
# ============================================================
# ✅ This reads the mempool_log.csv file created by mempool_logger.ipynb
# Handles missing file, bad lines, or empty datasets.

DATA_FILE = "mempool_log.csv"

try:
    df = pd.read_csv(DATA_FILE, on_bad_lines='skip', encoding='utf-8')
    if df.empty:
        print("⚠️ CSV file is empty. Run mempool_logger.ipynb first to collect data.")
    else:
        print(f"✅ Dataset loaded successfully with {len(df)} transactions.")
except FileNotFoundError:
    print("❌ ERROR: 'mempool_log.csv' not found. Run mempool_logger.ipynb first.")
    df = pd.DataFrame()
except Exception as e:
    print(f"⚠️ Error loading data: {e}")
    df = pd.DataFrame()


✅ Dataset loaded successfully with 20 transactions.


In [13]:
# ============================================================
# BLOCK 3: Clean and validate the data
# ============================================================
# ✅ Converts numeric columns, handles missing values, and removes duplicates.

if not df.empty:
    # Convert columns to numeric, replace invalid with 0
    for col in ["fee", "vsize"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Ensure ancestor column exists and has a default value
    if "ancestors" not in df.columns:
        df["ancestors"] = "None"

    df["ancestors"] = df["ancestors"].fillna("None")

    # Drop duplicate txids to avoid re-evaluation
    df.drop_duplicates(subset=["txid"], inplace=True)

    print(f"✅ Data cleaned successfully. {len(df)} unique transactions ready.")
else:
    print("⚠️ Skipping cleaning step — empty dataframe.")


✅ Data cleaned successfully. 20 unique transactions ready.


In [ ]:
# ============================================================
# BLOCK 4: Define the evaluation formula (Debug version)
# ============================================================
# ✅ Incorporates:
#   1. Normalization (Fee per vsize)
#   2. Sigmoid scaling (smooths large differences)
#   3. Ancestor factor (logarithmic weight)
#   4. Tunable constants alpha (scaling) and beta (ancestor strength)
# ✅ Logs intermediate results for debugging.

def evaluate_transaction(row, alpha=0.8, beta=1.2, debug=False):
    """
    Evaluates a transactions efficiency using normalized fee,
    sigmoid smoothing, and ancestor factor.
    
    Parameters:
      alpha (float): scaling constant (balances formula)
      beta (float): ancestor factor strength
      debug (bool): if True, prints each calculation step
    """
    try:
        fee = row.get("fee", 0)
        vsize = row.get("vsize", 0)
        ancestors = row.get("ancestors", "None")

        # Convert ancestor list into count
        ancestor_count = 0 if ancestors == "None" else len(ancestors.split(";"))

        if vsize <= 0:
            return 0  # avoid division by zero

        # --- Step 1: Normalization (fee per virtual size)
        norm_fee = fee / (vsize + 1)

        # --- Step 2: Sigmoid scaling ---
        # Keeps extreme values between 0–1 range
        sigmoid_value = 1 / (1 + exp(-norm_fee / 100))

        # --- Step 3: Ancestor factor (logarithmic weight)
        ancestor_factor = 1 + log1p(ancestor_count)

        # --- Step 4: Final formula ---
        efficiency_score = alpha * sigmoid_value * (beta ** ancestor_factor)

        if debug:
            print(f"TXID: {row.get('txid', 'unknown')}")
            print(f"  Fee: {fee}, vSize: {vsize}, Ancestors: {ancestor_count}")
            print(f"  Normalized Fee: {norm_fee:.4f}")
            print(f"  Sigmoid: {sigmoid_value:.4f}")
            print(f"  Ancestor Factor: {ancestor_factor:.4f}")
            print(f"  Efficiency Score: {efficiency_score:.4f}")
            print("-" * 40)

        return efficiency_score

    except Exception as e:
        print(f"⚠️ Error evaluating transaction: {e}")
        return 0


In [15]:
# ============================================================
# BLOCK 5: Apply formula and handle evaluation errors
# ============================================================
if not df.empty:
    try:
        # Apply formula to all rows
        df["efficiency_score"] = df.apply(
            lambda row: evaluate_transaction(row, alpha=0.8, beta=1.2, debug=False), axis=1
        )

        # Sort transactions by score
        df.sort_values(by="efficiency_score", ascending=False, inplace=True)

        print("✅ Formula applied successfully to all transactions.")
    except Exception as e:
        print(f"⚠️ Error applying formula: {e}")
else:
    print("⚠️ No transactions available to evaluate.")


✅ Formula applied successfully to all transactions.


In [16]:
# ============================================================
# BLOCK 6: Debug check for sample transactions
# ============================================================
# ✅ Runs the formula on first 3 transactions with debug=True for step-by-step output.

if not df.empty:
    print("\n🔍 Debugging first 3 transactions:")
    sample = df.head(3)
    for _, row in sample.iterrows():
        evaluate_transaction(row, alpha=0.8, beta=1.2, debug=True)
else:
    print("⚠️ No data to debug — ensure CSV is generated.")



🔍 Debugging first 3 transactions:
TXID: 2db813295b23969d36b2b12272c1819a87006e156789dc5e604452b54ea3bef7
  Fee: 1589, vSize: 1585, Ancestors: 17
  Normalized Fee: 1.0019
  Sigmoid: 0.5025
  Ancestor Factor: 3.8904
  Efficiency Score: 0.8171
----------------------------------------
TXID: 8b8e0f706f3ade201c49b2d51401e15945d47e8dbb4388558d936632194b8ae8
  Fee: 10081, vSize: 109, Ancestors: 1
  Normalized Fee: 91.6455
  Sigmoid: 0.7143
  Ancestor Factor: 1.6931
  Efficiency Score: 0.7781
----------------------------------------
TXID: 264f2c76bbebe9a499e8083af8be99fab256543bc1da1561b9c3767abda745e3
  Fee: 419, vSize: 347, Ancestors: 4
  Normalized Fee: 1.2040
  Sigmoid: 0.5030
  Ancestor Factor: 2.6094
  Efficiency Score: 0.6476
----------------------------------------


In [17]:
# ============================================================
# BLOCK 7: Display top-ranked transactions
# ============================================================
if not df.empty:
    print("\n🏆 Top 10 Efficient Transactions:")
    print(df[["txid", "fee", "vsize", "ancestors", "efficiency_score"]].head(10))
else:
    print("⚠️ Nothing to display — please run mempool_logger first.")



🏆 Top 10 Efficient Transactions:
                                                 txid    fee  vsize  \
17  2db813295b23969d36b2b12272c1819a87006e156789dc...   1589   1585   
7   8b8e0f706f3ade201c49b2d51401e15945d47e8dbb4388...  10081    109   
6   264f2c76bbebe9a499e8083af8be99fab256543bc1da15...    419    347   
0   8e76906085798561ef76c3584d36158cb170b215fa88fe...    532    260   
11  ee06f574cb262c748b83b8d4d65bc0ba38111172258570...    324    214   
12  26a8dc6438693a2e0f24a7af18e6f47baba67f11f664e1...    465    208   
4   19908adefab37dd34f139edf0977d845afbf05ad8a617f...    257    255   
19  64e414564fccf42dd220fddcc9c851f3c232c18c75c902...    701    112   
13  2e1bce91d0998ca9f47cc14854148dce4d15545576b19b...    257    109   
18  2915fc3569aef43318ad8ffaae37dc763de5d1602a2f17...    322    140   

                                            ancestors  efficiency_score  
17  f2a44c7d7fd071232234887f807818a8ed9d95289f5953...          0.817099  
7   24410fc31ddbb1849d6dd94f5535c434

In [18]:
# ===============================================================
# 6️⃣ Handle NaN or infinity issues
# ===============================================================
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

In [19]:
# ============================================================
# BLOCK 8: Save evaluated data into new CSV
# ============================================================
# ✅ Saves the results to a new file for reporting or visualization

OUTPUT_FILE = "evaluated_transactions.csv"

if not df.empty:
    try:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"✅ Evaluated dataset saved as '{OUTPUT_FILE}' ({len(df)} rows).")
    except Exception as e:
        print(f"⚠️ Error saving evaluated file: {e}")
else:
    print("⚠️ No evaluated data to save.")


✅ Evaluated dataset saved as 'evaluated_transactions.csv' (20 rows).
